# The Chain Rule and Backpropagation

Notebook for **Chapter 13** of the *math* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/math/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 The Chain Rule and Why Deep Learning Is Repeated Chain Rule

In [ ]:
import numpy as np

def num_deriv(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

print(round(num_deriv(lambda x: (3 * x + 1) ** 4, 1.0), 3))
print(round(num_deriv(lambda x: np.log(1 + np.exp(2 * x)), 0.0), 6))
print(round(num_deriv(lambda w: (w * 2 - 1) ** 2, 0.3), 4))

768.0
1.0
-1.6


## 13.2 Computational Graphs: Forward Pass, Backward Pass, Fan-Out

In [ ]:
# Forward and backward by hand for both graphs
x, y, z = -2.0, 5.0, -4.0
q = x + y
f = q * z
df = 1.0
dq = df * z            # multiply node: gradient to q is upstream * z
dz = df * q            # gradient to z is upstream * q
dx = dq * 1.0          # add node distributes
dy = dq * 1.0
print("graph 1:", q, f, dq, dz, dx, dy)

x, y = 2.0, 3.0
a = x * y
b = x + y
c = a * b
da, db = b, a                       # upstream is 1
dx = da * y + db * 1.0              # two routes into x: ACCUMULATE
dy = da * x + db * 1.0
print("graph 2:", a, b, c, da, db, dx, dy)
print("wrong (overwrite):", db * 1.0)

graph 1: 3.0 -12.0 -4.0 3.0 -4.0 -4.0
graph 2: 6.0 5.0 30.0 5.0 6.0 21.0 16.0
wrong (overwrite): 6.0


## 13.3 Backpropagation by Hand: One Neuron, Then a Two-Layer Network

In [ ]:
import numpy as np
sig = lambda z: 1 / (1 + np.exp(-z))

def loss(w, b, x=1.5, y=1.0):
    return (sig(w * x + b) - y) ** 2

x, w, b, y = 1.5, 0.8, -0.5, 1.0
z = w * x + b; a = sig(z); L = (a - y) ** 2
dL_da = 2 * (a - y); da_dz = a * (1 - a)
dL_dz = dL_da * da_dz
dL_dw, dL_db, dL_dx = dL_dz * x, dL_dz * 1.0, dL_dz * w
print("z a L      :", round(z, 4), round(a, 4), round(L, 4))
print("dL/da da/dz:", round(dL_da, 4), round(da_dz, 4))
print("dL/dz      :", round(dL_dz, 4))
print("dL/dw dL/db dL/dx:", round(dL_dw, 4), round(dL_db, 4), round(dL_dx, 4))

# numerical check of dL/dw and dL/db
h = 1e-6
print("numeric w,b:", round((loss(w + h, b) - loss(w - h, b)) / (2 * h), 4),
      round((loss(w, b + h) - loss(w, b - h)) / (2 * h), 4))

w2, b2 = w - 0.5 * dL_dw, b - 0.5 * dL_db
print("after step :", round(w2, 4), round(b2, 4), round(loss(w2, b2), 4))

z a L      : 0.7 0.6682 0.1101
dL/da da/dz: -0.6636 0.2217
dL/dz      : -0.1471
dL/dw dL/db dL/dx: -0.2207 -0.1471 -0.1177
numeric w,b: -0.2207 -0.1471
after step : 0.9104 -0.4264 0.079


In [ ]:
import numpy as np
sig = lambda z: 1 / (1 + np.exp(-z))

x = np.array([1.0, 2.0]); y = 0.0
W1 = np.array([[0.5, 0.25], [-0.6, 0.1]]); b1 = np.array([0.1, 0.05])
v = np.array([0.7, -0.4]); c = 0.2

def forward(W1, b1, v, c):
    z = W1 @ x + b1; h = np.maximum(z, 0); s = v @ h + c; a = sig(s)
    return z, h, s, a, 0.5 * (a - y) ** 2

z, h, s, a, L = forward(W1, b1, v, c)
print("z", z, "h", h, "s", round(s, 4), "a", round(a, 4), "L", round(L, 4))

ds = (a - y) * a * (1 - a)              # delta at the output node
dv, dc = ds * h, ds
dh = ds * v
dz = dh * (z > 0)                       # ReLU gate
dW1 = np.outer(dz, x); db1 = dz
dx = W1.T @ dz
print("ds", round(ds, 4), "dv", np.round(dv, 4), "dc", round(dc, 4))
print("dz", np.round(dz, 4) + 0.0)
print("dW1", (np.round(dW1, 4) + 0.0).tolist(), "db1", np.round(db1, 4) + 0.0)
print("dx", np.round(dx, 4))

z, h, s, a, L = forward(W1 - dW1, b1 - db1, v - dv, c - dc)
print("after step: a", round(a, 4), "L", round(L, 4))

z [ 1.1  -0.35] h [1.1 0. ] s 0.97 a 0.7251 L 0.2629
ds 0.1445 dv [0.159 0.   ] dc 0.1445
dz [0.1012 0.    ]
dW1 [[0.1012, 0.2023], [0.0, 0.0]] db1 [0.1012 0.    ]
dx [0.0506 0.0253]
after step: a 0.5799 L 0.1681


## 13.4 Vectorised Backpropagation: Shapes, Softmax and Activations

In [ ]:
import numpy as np
W = np.array([[1., 0., -1.], [2., 1., 0.]])
X = np.array([[1., 2., 3.], [0., 1., -1.]])
D = np.array([[1., -1.], [0.5, 2.]])
print("dW", (D.T @ X).tolist())
print("db", D.sum(axis=0).tolist())
print("dX", (D @ W).tolist())

dW [[1.0, 2.5, 2.5], [-1.0, 0.0, -5.0]]
db [1.5, 1.0]
dX [[-1.0, -1.0, -1.0], [4.5, 2.0, -0.5]]


In [ ]:
import numpy as np
z = np.array([2.0, 1.0, 0.1]); y = np.array([1.0, 0.0, 0.0])

def ce(z):
    e = np.exp(z - z.max()); p = e / e.sum()
    return -np.log(p[0])

e = np.exp(z - z.max()); p = e / e.sum()
print("p     ", np.round(p, 4), " loss", round(ce(z), 4))
print("p - y ", np.round(p - y, 4), " sum", round((p - y).sum(), 12))
h = 1e-6
num = [(ce(z + h * np.eye(3)[i]) - ce(z - h * np.eye(3)[i])) / (2 * h) for i in range(3)]
print("numeric", np.round(num, 4))

p      [0.659  0.2424 0.0986]  loss 0.417
p - y  [-0.341   0.2424  0.0986]  sum 0.0
numeric [-0.341   0.2424  0.0986]


In [ ]:
import numpy as np
w, x, b, up = 2.0, 0.5, -0.5, 3.0
a = np.tanh(w * x + b)
dz = up * (1 - a ** 2)
print(round(a, 4), round(dz, 4), round(dz * x, 4), round(dz, 4), round(dz * w, 4))

f = lambda w, b, x: 3.0 * np.tanh(w * x + b)      # a loss whose dL/da is exactly 3
h = 1e-6
print(round((f(w + h, b, x) - f(w - h, b, x)) / (2 * h), 4))

0.4621 2.3593 1.1797 2.3593 4.7187
1.1797


## 13.5 A Neural Network from Scratch in NumPy, and Gradient Checking

In [ ]:
import numpy as np

def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))   # subtract max for stability
    return E / E.sum(axis=1, keepdims=True)

class MLP:
    def __init__(self, sizes, act="relu", seed=0):
        rng = np.random.default_rng(seed)
        self.act = act
        self.W, self.b = [], []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            gain = 2.0 if act == "relu" else 1.0          # He for relu, Xavier-like for tanh
            self.W.append(rng.normal(0, np.sqrt(gain / n_in), (n_out, n_in)))
            self.b.append(np.zeros(n_out))

    def f(self, Z):
        return np.maximum(Z, 0) if self.act == "relu" else np.tanh(Z)

    def df(self, Z, A):
        return (Z > 0).astype(float) if self.act == "relu" else 1 - A**2

    def forward(self, X):
        self.As, self.Zs = [X], []
        A = X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            Z = A @ W.T + b                      # (N, n_out)
            self.Zs.append(Z)
            A = self.f(Z) if i < len(self.W) - 1 else Z
            self.As.append(A)
        return softmax(self.As[-1])              # class probabilities

    def loss(self, P, y):
        return -np.log(P[np.arange(len(y)), y]).mean()

    def backward(self, P, y):
        N = len(y)
        dZ = P.copy()
        dZ[np.arange(N), y] -= 1                 # p - y (one-hot)
        dZ /= N                                  # mean over the batch
        gW, gb = [None] * len(self.W), [None] * len(self.W)
        for i in reversed(range(len(self.W))):
            gW[i] = dZ.T @ self.As[i]            # (n_out, n_in)
            gb[i] = dZ.sum(axis=0)
            if i > 0:
                dA = dZ @ self.W[i]              # (N, n_in)
                dZ = dA * self.df(self.Zs[i-1], self.As[i])
        return gW, gb

    def train(self, X, y, lr=0.5, epochs=1000, every=250):
        hist = []
        for ep in range(epochs + 1):
            P = self.forward(X)
            L = self.loss(P, y)
            hist.append(L)
            if every and ep % every == 0:
                acc = (P.argmax(axis=1) == y).mean()
                print(f"epoch {ep:5d}  loss {L:.4f}  acc {acc:.3f}")
            gW, gb = self.backward(P, y)
            for i in range(len(self.W)):
                self.W[i] -= lr * gW[i]
                self.b[i] -= lr * gb[i]
        return hist

# ---------- 1. gradient check on a tiny tanh network ----------
def numeric_grad(net, X, y, h=1e-5):
    grads = []
    for arr in net.W + net.b:
        g = np.zeros_like(arr)
        for idx in np.ndindex(*arr.shape):
            old = arr[idx]
            arr[idx] = old + h; Lp = net.loss(net.forward(X), y)
            arr[idx] = old - h; Lm = net.loss(net.forward(X), y)
            arr[idx] = old
            g[idx] = (Lp - Lm) / (2 * h)
        grads.append(g)
    return grads

def rel_error(ga, gn):
    a = np.concatenate([g.ravel() for g in ga]); n = np.concatenate([g.ravel() for g in gn])
    return np.linalg.norm(a - n) / (np.linalg.norm(a) + np.linalg.norm(n))

rng = np.random.default_rng(5)
Xs = rng.normal(size=(5, 3)); ys = np.array([0, 2, 1, 1, 0])
net = MLP([3, 4, 3], act="tanh", seed=1)
P = net.forward(Xs); gW, gb = net.backward(P, ys)
gn = numeric_grad(net, Xs, ys)
print("relative error, correct backward:", f"{rel_error(gW + gb, gn):.1e}")
bad = [g * len(ys) for g in gW + gb]             # bug: forgot the 1/N
print("relative error, buggy backward  :", f"{rel_error(bad, gn):.2f}")

# ---------- 2. XOR ----------
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1.0]]); y = np.array([0, 1, 1, 0])
net = MLP([2, 4, 2], act="tanh", seed=1)
net.train(X, y, lr=0.5, epochs=1000, every=250)
print("P(class 1) for the four XOR inputs:", np.round(net.forward(X)[:, 1], 3))

# ---------- 3. three-arm spiral ----------
def spiral(n=100, k=3, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n * k, 2)); y = np.zeros(n * k, dtype=int)
    for j in range(k):
        r = np.linspace(0.05, 1, n)
        t = np.linspace(j * 4, (j + 1) * 4, n) + rng.normal(0, 0.2, n)
        X[j*n:(j+1)*n] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[j*n:(j+1)*n] = j
    return X, y

Xtr, ytr = spiral(seed=0); Xte, yte = spiral(seed=1)
net = MLP([2, 64, 3], act="relu", seed=0)
net.train(Xtr, ytr, lr=1.0, epochs=1000, every=250)
test_acc = (net.forward(Xte).argmax(axis=1) == yte).mean()
print("test accuracy on fresh spiral noise:", round(test_acc, 3))

relative error, correct backward: 4.5e-11
relative error, buggy backward  : 0.67
epoch     0  loss 0.7162  acc 0.500
epoch   250  loss 0.0218  acc 1.000
epoch   500  loss 0.0065  acc 1.000
epoch   750  loss 0.0036  acc 1.000
epoch  1000  loss 0.0024  acc 1.000
P(class 1) for the four XOR inputs: [0.001 0.998 0.997 0.004]
epoch     0  loss 1.1686  acc 0.227
epoch   250  loss 0.0818  acc 1.000
epoch   500  loss 0.0411  acc 1.000
epoch   750  loss 0.0266  acc 1.000
epoch  1000  loss 0.0193  acc 1.000
test accuracy on fresh spiral noise: 0.997


In [ ]:
w0 = 1.0
for lr in [0.01, 0.1, 0.5, 1.0, 1.1]:
    w = w0
    for step in range(10):
        w = w - lr * 2 * w          # gradient of w**2 is 2w
    print(f"lr {lr:5.2f}  factor {1 - 2*lr:6.2f}  w after 10 steps {w:9.4f}")

lr  0.01  factor   0.98  w after 10 steps    0.8171
lr  0.10  factor   0.80  w after 10 steps    0.1074
lr  0.50  factor   0.00  w after 10 steps    0.0000
lr  1.00  factor  -1.00  w after 10 steps    1.0000
lr  1.10  factor  -1.20  w after 10 steps    6.1917


## 13.6 Reverse-Mode vs Forward-Mode Autodiff, and a Micro Autograd Engine

In [ ]:
class Dual:
    """A number that carries (value, derivative)."""
    def __init__(self, v, d=0.0):
        self.v, self.d = v, d
    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.v + o.v, self.d + o.d)
    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.v * o.v, self.d * o.v + self.v * o.d)   # product rule

def f(x):
    return x[0]*x[1] + x[1]*x[2] + x[2]*x[3] + x[3]*x[0]

point = [1.0, 2.0, 3.0, 4.0]
grad, sweeps = [], 0
for i in range(4):                                    # one sweep per input
    xs = [Dual(p, 1.0 if j == i else 0.0) for j, p in enumerate(point)]
    grad.append(f(xs).d)
    sweeps += 1
print("forward-mode gradient:", grad, " sweeps:", sweeps)

n = 1e8                                              # parameters in a modest network
seconds = n * 1.0                                    # one second per forward sweep
print("forward mode for 1e8 weights:", round(seconds / (3600 * 24 * 365), 2), "years")

forward-mode gradient: [6.0, 4.0, 6.0, 4.0]  sweeps: 4
forward mode for 1e8 weights: 3.17 years


In [ ]:
import math

class Value:
    def __init__(self, data, parents=()):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad                       # add: distribute
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad          # multiply: swap
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):                               # k is a plain number
        out = Value(self.data ** k, (self,))
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,))
        def _backward():
            self.grad += out.data * out.grad            # d exp(z) = exp(z)
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,))
        def _backward():
            self.grad += (self.data > 0) * out.grad     # the gate
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * (other ** -1)
    def __rtruediv__(self, other): return other * (self ** -1)

    def backward(self):
        topo, seen = [], set()
        def build(v):                                   # depth-first topological sort
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                topo.append(v)                          # appended AFTER its parents
        build(self)
        self.grad = 1.0                                 # d output / d output
        for v in reversed(topo):                        # consumers before producers
            v._backward()

def sigmoid(z):
    return 1 / (1 + (-z).exp())

# 1. a tanh neuron: n = x1*w1 + x2*w2 + b, o = tanh(n)
x1, x2, w1, w2 = Value(2.0), Value(0.0), Value(-3.0), Value(1.0)
b = Value(6.8813735870195432)
o = (x1 * w1 + x2 * w2 + b).tanh()
o.backward()
print("tanh neuron o:", round(o.data, 4), " grads x1 w1 x2 w2:", round(x1.grad, 4), round(w1.grad, 4), round(x2.grad, 4), round(w2.grad, 4))

# 2. fan-out graph c = (x*y)(x+y) at x=2, y=3
x, y = Value(2.0), Value(3.0)
c = (x * y) * (x + y)
c.backward()
print("fan-out c:", c.data, " dx dy:", x.grad, y.grad)

# 3. the sigmoid neuron of Section 13.3
w, b, xin = Value(0.8), Value(-0.5), Value(1.5)
L = (sigmoid(w * xin + b) - 1) ** 2
L.backward()
print("sigmoid neuron L:", round(L.data, 4), " dw db dx:", round(w.grad, 4), round(b.grad, 4), round(xin.grad, 4))

# 4. the two-layer network of Section 13.3
xs = [Value(1.0), Value(2.0)]
W1 = [[Value(0.5), Value(0.25)], [Value(-0.6), Value(0.1)]]
b1 = [Value(0.1), Value(0.05)]
v = [Value(0.7), Value(-0.4)]; c0 = Value(0.2)
h = [(W1[i][0] * xs[0] + W1[i][1] * xs[1] + b1[i]).relu() for i in range(2)]
a = sigmoid(v[0] * h[0] + v[1] * h[1] + c0)
L = 0.5 * (a - 0) ** 2
L.backward()
print("MLP L:", round(L.data, 4))
print("dW1 row1:", round(W1[0][0].grad, 4), round(W1[0][1].grad, 4), " row2:", W1[1][0].grad, W1[1][1].grad)
print("dv:", round(v[0].grad, 4), v[1].grad, " dc:", round(c0.grad, 4), " dx:", round(xs[0].grad, 4), round(xs[1].grad, 4))

# 5. forward-mode example from above, reverse mode, ONE backward call
p = [Value(t) for t in (1.0, 2.0, 3.0, 4.0)]
f = p[0]*p[1] + p[1]*p[2] + p[2]*p[3] + p[3]*p[0]
f.backward()
print("reverse-mode gradient:", [q.grad for q in p], " backward calls: 1")

tanh neuron o: 0.7071  grads x1 w1 x2 w2: -1.5 1.0 0.5 0.0
fan-out c: 30.0  dx dy: 21.0 16.0
sigmoid neuron L: 0.1101  dw db dx: -0.2207 -0.1471 -0.1177
MLP L: 0.2629
dW1 row1: 0.1012 0.2023  row2: 0.0 0.0
dv: 0.159 0.0  dc: 0.1445  dx: 0.0506 0.0253
reverse-mode gradient: [6.0, 4.0, 6.0, 4.0]  backward calls: 1


In [ ]:
# ---- PyTorch equivalent (NOT RUN HERE: torch is not installed in this environment) ----
# import torch
# x1 = torch.tensor(2.0, requires_grad=True);  w1 = torch.tensor(-3.0, requires_grad=True)
# x2 = torch.tensor(0.0, requires_grad=True);  w2 = torch.tensor(1.0, requires_grad=True)
# b  = torch.tensor(6.8813735870195432, requires_grad=True)
# o = torch.tanh(x1 * w1 + x2 * w2 + b)
# o.backward()                                    # fills .grad of every leaf
# print(x1.grad.item(), w1.grad.item(), x2.grad.item(), w2.grad.item())
#
# Typical training step in PyTorch (also not run here):
# model = torch.nn.Sequential(torch.nn.Linear(2, 64), torch.nn.ReLU(), torch.nn.Linear(64, 3))
# opt = torch.optim.SGD(model.parameters(), lr=1.0)
# loss_fn = torch.nn.CrossEntropyLoss()           # softmax + cross-entropy fused (p - y backward)
# for epoch in range(1000):
#     opt.zero_grad()                             # gradients ACCUMULATE, so reset them
#     loss = loss_fn(model(X), y)                 # forward builds the graph
#     loss.backward()                             # backward sweeps it
#     opt.step()                                  # theta -= lr * grad
# with torch.no_grad(): ...                       # evaluation: no graph, less memory
print("expected from the hand calculation: -1.5 1.0 0.5 0.0   (x1, w1, x2, w2)")

expected from the hand calculation: -1.5 1.0 0.5 0.0   (x1, w1, x2, w2)


## 13.7 Vanishing and Exploding Gradients, Initialisation, Residuals and Batch Norm

In [ ]:
import numpy as np

def delta_norms(act, std, L=20, n=100, N=64, seed=0):
    rng = np.random.default_rng(seed)
    Ws = [rng.normal(0, std, (n, n)) for _ in range(L)]
    a = rng.normal(size=(N, n)); As = [a]; Zs = []
    for W in Ws:
        z = a @ W.T; Zs.append(z)
        a = 1/(1+np.exp(-z)) if act == "sigmoid" else np.maximum(z, 0) if act == "relu" else np.tanh(z)
        As.append(a)
    d = np.ones((N, n)); out = []
    for i in reversed(range(L)):
        if act == "sigmoid": d = d * As[i+1] * (1 - As[i+1])
        elif act == "relu": d = d * (Zs[i] > 0)
        else: d = d * (1 - As[i+1]**2)
        out.append(np.linalg.norm(d)); d = d @ Ws[i]
    return out[::-1]                                  # index 0 = layer closest to the input

n = 100
cases = [("sigmoid, Xavier", "sigmoid", np.sqrt(1/n)), ("relu, std 0.05", "relu", 0.05),
         ("relu, std 0.3", "relu", 0.3), ("relu, He", "relu", np.sqrt(2/n)), ("tanh, Xavier", "tanh", np.sqrt(1/n))]
print(f"{'setting':18s} {'layer 1':>10s} {'layer 10':>10s} {'layer 20':>10s}")
for name, act, std in cases:
    g = delta_norms(act, std)
    print(f"{name:18s} {g[0]:10.1e} {g[9]:10.1e} {g[19]:10.1e}")

rng = np.random.default_rng(0)
for name, var in [("Var(w) = 1/n", 1.0), ("Var(w) = 2/n (He)", 2.0)]:
    a = rng.normal(size=(1000, 500))
    for _ in range(20):
        a = np.maximum(a @ rng.normal(0, np.sqrt(var / 500), (500, 500)).T, 0)
    print(name, "RMS of layer-20 activations:", f"{np.sqrt((a**2).mean()):.4f}")

setting               layer 1   layer 10   layer 20
sigmoid, Xavier       1.7e-11    9.6e-06    1.9e+01
relu, std 0.05        4.0e-07    4.1e-03    5.9e+01
relu, std 0.3         2.4e+08    2.5e+05    5.9e+01
relu, He              1.5e+02    1.3e+02    5.9e+01
tanh, Xavier          1.4e+01    5.4e+01    7.8e+01
Var(w) = 1/n RMS of layer-20 activations: 0.0008
Var(w) = 2/n (He) RMS of layer-20 activations: 0.7336


In [ ]:
import numpy as np
rng = np.random.default_rng(1)
n, L = 50, 30
Js = [rng.normal(0, 0.05, (n, n)) for _ in range(L)]      # small layer Jacobians
u = np.ones(n) / np.sqrt(n)                                # gradient arriving at the top, norm 1
plain, res = u.copy(), u.copy()
for J in Js:
    plain = J.T @ plain                    # plain network: multiply by J
    res = (np.eye(n) + J).T @ res          # residual network: multiply by (I + J)
print("plain 30 layers   :", f"{np.linalg.norm(plain):.1e}")
print("residual 30 layers:", f"{np.linalg.norm(res):.2f}")

plain 30 layers   : 3.5e-14
residual 30 layers: 5.22


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = rng.normal(size=6) * 2 + 1
g = rng.normal(size=6)                    # upstream gradient dL/d xhat, with L = sum(g * xhat)
eps = 1e-5

def bn(x):
    return (x - x.mean()) / np.sqrt(x.var() + eps)

xhat = bn(x); sd = np.sqrt(x.var() + eps); N = len(x)
dx = (N * g - g.sum() - xhat * (g * xhat).sum()) / (N * sd)      # full batchnorm backward
naive = g / sd                                                    # pretends mean and variance are constants

L = lambda x: (g * bn(x)).sum()
h = 1e-6
num = np.array([(L(x + h*np.eye(N)[i]) - L(x - h*np.eye(N)[i])) / (2*h) for i in range(N)])
print("full formula :", np.round(dx, 4))
print("numeric      :", np.round(num, 4))
print("naive (wrong):", np.round(naive, 4))
print("max error full vs numeric:", f"{np.abs(dx - num).max():.1e}", " |sum of dx|:", abs(round(dx.sum(), 10)))

# gradient clipping by global norm
grads = [np.array([30.0, -40.0]), np.array([0.0, 120.0])]
total = np.sqrt(sum((q**2).sum() for q in grads))
scale = min(1.0, 5.0 / total)
print("global norm", round(total, 2), " scale", round(scale, 5), " clipped:", [np.round(q * scale, 4).tolist() for q in grads])

full formula : [ 1.8412  1.2868 -0.7335 -1.642  -0.9502  0.1977]
numeric      : [ 1.8412  1.2868 -0.7335 -1.642  -0.9502  0.1977]
naive (wrong): [ 1.7648  1.2818 -0.9524 -1.7126 -0.8435  0.0559]
max error full vs numeric: 2.5e-10  |sum of dx|: 0.0
global norm 130.0  scale 0.03846  clipped: [[1.1538, -1.5385], [0.0, 4.6154]]


## 13.8 The Expert Layer: JVPs and VJPs, Memory, Checkpointing, Second Order, BPTT

In [ ]:
import numpy as np

# VJP of softmax without the Jacobian
z = np.array([2.0, 1.0, 0.1]); u = np.array([1.0, -2.0, 0.5])
p = np.exp(z) / np.exp(z).sum()
J = np.diag(p) - np.outer(p, p)
print("u^T J (explicit Jacobian)  :", np.round(u @ J, 4))
print("p * (u - p.u) (no Jacobian):", np.round(p * (u - p @ u), 4))

# memory of stored activations
layers, batch, width, nbytes = 50, 512, 4096, 4
per_layer = batch * width * nbytes * 2
print("MiB per layer:", per_layer / 2**20, " all layers GiB:", round(layers * per_layer / 2**30, 3))

# checkpointing: stored tensors = L/s checkpoints + s live activations
L = 100
for s in [1, 5, 10, 20, 50, 100]:
    print(f"segment {s:3d}: stored activations {L // s + s}")

u^T J (explicit Jacobian)  : [ 0.5118 -0.539   0.0273]
p * (u - p.u) (no Jacobian): [ 0.5118 -0.539   0.0273]
MiB per layer: 16.0  all layers GiB: 0.781
segment   1: stored activations 101
segment   5: stored activations 25
segment  10: stored activations 20
segment  20: stored activations 25
segment  50: stored activations 52
segment 100: stored activations 101


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(size=(50, 3))
y = (X @ np.array([1., -2., 0.5]) + rng.normal(size=50) > 0) * 2.0 - 1     # labels in {-1, +1}
sig = lambda t: 1 / (1 + np.exp(-t))

def grad(w):                                          # gradient of mean log(1 + exp(-y x.w))
    return X.T @ (-y * sig(-y * (X @ w))) / len(y)

w = np.array([0.3, -0.1, 0.2]); v = np.array([1.0, 2.0, -1.0])
s = sig(X @ w)
H = X.T @ (X * (s * (1 - s))[:, None]) / len(y)       # exact Hessian
eps = 1e-5
hv_fd = (grad(w + eps * v) - grad(w - eps * v)) / (2 * eps)   # no Hessian needed
print("exact H v     :", np.round(H @ v, 6))
print("gradient diff :", np.round(hv_fd, 6))

exact H v     : [ 0.317444  0.358911 -0.173782]
gradient diff : [ 0.317444  0.358911 -0.173782]


In [ ]:
import numpy as np

def chain_grad(w, T=30, tanh_act=True):
    """d h_T / d h_0 for h_t = f(w * h_{t-1} + 0.1), as a product of local factors."""
    h, jac = 0.5, 1.0
    for _ in range(T):
        h = np.tanh(w * h + 0.1) if tanh_act else w * h + 0.1
        jac *= w * (1 - h**2) if tanh_act else w
    return jac

def numeric(w, T=30, e=1e-6):
    def run(h0):
        h = h0
        for _ in range(T): h = np.tanh(w * h + 0.1)
        return h
    return (run(0.5 + e) - run(0.5 - e)) / (2 * e)

print(f"{'w':>5s} {'linear w^30':>12s} {'tanh chain':>12s} {'tanh numeric':>13s}")
for w in [0.9, 1.1, 1.5]:
    print(f"{w:5.1f} {chain_grad(w, tanh_act=False):12.3e} {chain_grad(w):12.3e} {numeric(w):13.3e}")

w  linear w^30   tanh chain  tanh numeric
  0.9    4.239e-02    7.132e-06     7.132e-06
  1.1    1.745e+01    4.907e-08     4.907e-08
  1.5    1.918e+05    1.443e-15     0.000e+00


## Chapter recap

## Practice set

Problem 13.1 Differentiate y = (2x + 1)³ at x = 1 with the chain rule.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/math/ch13.html